# Banking agent: VAE + GAN + LoRA (PEFT) on Google Colab

Runs the three new modules. Use a **GPU runtime** (Runtime -> Change runtime type -> T4 GPU).

1. Setup (upload the project, install packages)
2. VAE + GAN experiment (`src.augment`) - works on CPU or GPU
3. LoRA fine-tuning of Qwen2.5-1.5B-Instruct (`src.peft_lora`) - needs the GPU
4. Compare RAG vs PEFT vs hybrid on the held-out subset
5. Optional: RAG with the synthetic pool, and the out-of-scope guard

## 1. Setup
Clones the repository from GitHub into the Colab session and installs the packages. Push your latest changes first: the clone takes what is on GitHub. (Fallback: if a `banking-agent.zip` is uploaded to `/content`, it is used instead.)

In [ ]:
import os, zipfile
if os.path.exists('/content/banking-agent.zip'):
    zipfile.ZipFile('/content/banking-agent.zip').extractall('/content')
elif not os.path.exists('/content/banking-agent'):
    !git clone https://github.com/2025paml556-bhoomika/banking-agent.git /content/banking-agent
%cd /content/banking-agent
!pip -q install transformers peft accelerate jsonschema scikit-learn scipy pandas gradio
# optional, only for QLoRA (4-bit): !pip -q install bitsandbytes

In [ ]:
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. VAE + GAN
Trains the conditional text VAE and the latent GAN, generates synthetic messages, filters them, and measures the effect on a classifier (full data and 10 % low-resource). Saves `models/vae.pt`, `models/gan.pt`, `data/synthetic_*.csv`, `evaluation/results/vae_gan_results.json`.

In [ ]:
!python -m src.augment --vae-epochs 30 --gan-steps 3000 --per-intent 30 --low-seeds 3

In [ ]:
import json, pandas as pd
R = json.load(open('evaluation/results/vae_gan_results.json'))
print(json.dumps({k: R[k] for k in ['gen_stats','latent_frechet','kept_after_filter','downstream_full','knn5_vote','guard']}, indent=2))
print(pd.read_csv('evaluation/results/samples_vae.csv').head(10).to_string())
print(pd.read_csv('evaluation/results/samples_gan.csv').head(10).to_string())

## 3. LoRA fine-tuning (PEFT)
Trains adapters on `data/train.csv` only. One epoch of 10,003 rows at batch 16 is about 626 steps. The adapter is saved to `models/lora-intent`; the base model stays frozen.

In [ ]:
!python -m src.peft_lora train --model Qwen/Qwen2.5-1.5B-Instruct --out models/lora-intent --epochs 1 --batch-size 16 --lr 2e-4 --r 16 --alpha 32

In [ ]:
import json; print(json.dumps(json.load(open('evaluation/results/peft_summary.json')), indent=2))
print(json.load(open('models/lora-intent/train_log.json'))['trainable'], 'trainable parameters')

## 4. Compare the classifiers on the 100 held-out rows
RAG (default), PEFT, and the hybrid (PEFT first, RAG fallback). Each run is a separate process because `src/llm.py` loads the model at import time.

In [ ]:
!CLASSIFIER=retrieval python -m evaluation.final_eval
!ADAPTER_PATH=models/lora-intent CLASSIFIER=peft python -m evaluation.final_eval
!ADAPTER_PATH=models/lora-intent CLASSIFIER=peft+retrieval python -m evaluation.final_eval

In [ ]:
import pandas as pd
for f in ['results_heldout','results_heldout_peft','results_heldout_peft+retrieval']:
    d = pd.read_csv(f'evaluation/results/{f}.csv'); print(f, 'accuracy', d['correct'].mean(), 'no valid intent', int((d['predicted'].isna() | (d['predicted']=='')).sum()))

## 5. Optional: RAG with the synthetic pool, and the VAE guard

In [ ]:
!RETRIEVAL_POOL=vae+gan python -m src.retrieval

In [ ]:
!python - <<'EOF'
from src.guard import vae_score
for m in ['I lost my card', 'What is the weather in Paris?', 'How long does a transfer take?', 'Write me a poem']:
    print(m, vae_score(m))
EOF